In [ ]:
from pathlib import Path
import sys

# Localizar la raiz del repositorio desde Jupyter o terminal
RAIZ_REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks").is_dir() and (p / "datos").is_dir()),
    None
)
if RAIZ_REPO is None:
    raise FileNotFoundError("Ejecuta desde el repositorio caso-itam")

sys.path.insert(0, str(RAIZ_REPO / "src"))


# Selección de covariables y EBLUP de Fay-Herriot

In [ ]:
# --- Rutas -------------------------------------------------------------------
# Los insumos viven en dos carpetas: lo de los días 2 a 5 en "Datos procesados"
# y lo de supervivencia (días 6 a 8) en "EDA/ENVIPE/salidas".
BASE = RAIZ_REPO
RUTA_DATOS  = BASE / "datos" / "procesados"
RUTA_FGJ    = BASE / "resultados" / "supervivencia"

# Las salidas de hoy van a una carpeta propia para no mezclarlas
RUTA_SALIDA = BASE / "resultados" / "dia09_10"
RUTA_SALIDA.mkdir(exist_ok=True)

# Verificación de que las carpetas existen antes de intentar leer
for r in [RUTA_DATOS, RUTA_FGJ]:
    print(("OK      " if r.exists() else "NO EXISTE ") + str(r))

maestro      = pd.read_parquet(RUTA_DATOS / "maestro_alcaldia.parquet")
directos     = pd.read_parquet(RUTA_DATOS / "estimaciones_directas_v2.parquet")
diccionario  = pd.read_csv(RUTA_DATOS / "diccionario_maestro.csv")
cov_fgj      = pd.read_parquet(RUTA_FGJ / "covariables_fgj_alcaldia_anio.parquet")
completitud  = pd.read_parquet(RUTA_FGJ / "factor_completitud.parquet")

print("pandas", pd.__version__)

In [ ]:
# verificación contra las cifras de los días 4 y 5

# maestro: 16 alcaldías × 5 años = 80 filas, 53 columnas, sin nulos (día 4)
# directos: 2 geografías × 5 años × 16 alcaldías × 3 parámetros = 480 (día 5)
# cov_fgj: 80 filas alcaldía-año (días 6-8)
for nombre, df in [("maestro", maestro), ("directos", directos),
                   ("cov_fgj", cov_fgj), ("completitud", completitud)]:
    print(f"{nombre:12s} forma={df.shape}  nulos={int(df.isna().sum().sum())}")

assert maestro.shape == (80, 53), "El maestro no tiene la forma del día 4"
assert len(directos) == 480,      "Las estimaciones directas no tienen 480 filas"

# Nombres de columnas: los necesito para escribir las celdas siguientes
print("\nColumnas de directos:\n", list(directos.columns))
print("\nColumnas de cov_fgj:\n", list(cov_fgj.columns))
print("\nColumnas de completitud:\n", list(completitud.columns))
print("\nPrimeras filas de directos:")
display(directos.head())

In [ ]:
# ¿dónde están los nulos?, ¿qué celdas entran al modelo?,
#          y verificación de los conteos de CV del día 5

# --- 1. Nulos por columna -----------------------------------------------------
nulos = directos.isna().sum()
print("Nulos por columna:\n", nulos[nulos > 0], "\n")

# --- 2. Tipos de celda y tratamiento decidido el día 5 ------------------------
print(pd.crosstab([directos["tipo_celda"], directos["tratamiento_modelo"]],
                  directos["param"]), "\n")

# Filas con algún nulo: ¿coinciden con las celdas especiales?
con_nulo = directos[directos.isna().any(axis=1)]
print("Filas con nulos por tipo de celda y parámetro:")
print(pd.crosstab(con_nulo["tipo_celda"], con_nulo["param"]), "\n")

# --- 3. Verificación de la varianza de trabajo --------------------------------
# var_theta_asen debe ser 1 / (4 · n_ef_suav) en todas las filas donde exista.
# Si esto se cumple, sabemos exactamente qué psi_d va a usar el Fay-Herriot.
ok = directos["var_theta_asen"].notna()
dif = (directos.loc[ok, "var_theta_asen"]
       - 1 / (4 * directos.loc[ok, "n_ef_suav"])).abs().max()
print(f"Máxima diferencia var_theta_asen vs 1/(4·n_ef_suav): {dif:.2e}")
print("deff_suav por parámetro y geografía:")
print(directos.groupby(["geografia", "param"])["deff_suav"].unique(), "\n")

# --- 4. Reproducción de los conteos del día 5 ---------------------------------
# Esperado (ocurrencia): R = 51, d = 32, k = 9 celdas con CV > 30% de 80
print("Celdas con CV > 30% por geografía y parámetro:")
print(directos.assign(cv_alto=directos["cv"] > 0.30)
              .pivot_table(index="geografia", columns="param",
                           values="cv_alto", aggfunc="sum"), "\n")

# --- 5. Llaves para unir con las covariables ----------------------------------
# directos usa 'alc' ('002'); cov_fgj y completitud usan 'cve_mun'.
# Hay que ver qué usa el maestro y si los formatos coinciden.
print("Columnas del maestro:\n", list(maestro.columns), "\n")
print("Ejemplos de llaves:")
print("  directos.alc     ", directos["alc"].unique()[:4], directos["alc"].dtype)
print("  cov_fgj.cve_mun  ", cov_fgj["cve_mun"].unique()[:4], cov_fgj["cve_mun"].dtype)

In [ ]:
# base de análisis (geografía de ocurrencia) y correlaciones

# La geografía principal es ocurrencia; residencia queda como robustez.
# Unimos a cada celda directa sus covariables por (alcaldía, año de referencia).

COV_FIJAS = ["z_prop_fem_18", "z_prop_ocupada_12", "z_log_densidad",
             "z_eje_socio_1", "z_eje_socio_2", "z_agencias_mp_100k",
             "z_dist_mp_km"]                       # censo 2020: no cambian
COV_TIEMPO = ["z_sesnsp_nucleo", "z_prop_7d_std", "z_prop_digital_robo"]
CANDIDATAS = COV_FIJAS + COV_TIEMPO

# --- Covariables FGJ: estandarizar igual que las demás ------------------------
# prop_7d_std ya está "estandarizada" en el sentido de composición de delitos,
# pero no está en escala z. La pasamos a z sobre las 80 filas para que los
# coeficientes sean comparables y el índice de condición tenga sentido.
fgj = cov_fgj[["cve_mun", "anio_ref", "prop_7d_std", "prop_digital_robo"]].copy()
for v in ["prop_7d_std", "prop_digital_robo"]:
    fgj["z_" + v] = (fgj[v] - fgj[v].mean()) / fgj[v].std(ddof=1)

covs = (maestro[["cve_mun", "anio_ref", "nombre"] + COV_FIJAS + ["z_sesnsp_nucleo"]]
        .merge(fgj, on=["cve_mun", "anio_ref"], how="inner", validate="1:1"))
assert len(covs) == 80, "La unión de covariables no dio 80 filas"

# --- Base de análisis: directos de ocurrencia + covariables -------------------
base = (directos.query("geografia == 'ocurrencia'")
        .merge(covs.drop(columns="nombre"), left_on=["alc", "anio_ref"],
               right_on=["cve_mun", "anio_ref"], how="left", validate="m:1"))
assert len(base) == 240 and base[CANDIDATAS].notna().all().all()

print("Celdas por parámetro y tipo (ocurrencia):")
print(pd.crosstab(base["tipo_celda"], base["param"]), "\n")

# --- Matriz de correlaciones de las 10 candidatas -----------------------------
# Se calcula sobre las 80 filas alcaldía-año (una vez, no por parámetro):
# las covariables son las mismas para d, k y R.
corr = covs[CANDIDATAS].corr()
print("Pares con |r| > 0.70:")
pares = [(a, b, corr.loc[a, b]) for a, b in itertools.combinations(CANDIDATAS, 2)
         if abs(corr.loc[a, b]) > 0.70]
for a, b, r in sorted(pares, key=lambda x: -abs(x[2])):
    print(f"  {a:22s} {b:22s} {r:+.2f}")

# --- ¿Cuánto varían en el tiempo las covariables que cambian por año? ---------
# Fracción de la varianza total que es variación DENTRO de cada alcaldía.
# Esto es lo que mañana puede explicar la parte temporal en el Rao-Yu.
print("\nFracción de varianza dentro de alcaldía (temporal):")
for v in COV_TIEMPO:
    dentro = covs.groupby("cve_mun")[v].transform(lambda s: s - s.mean()).var()
    print(f"  {v:22s} {dentro / covs[v].var():.2f}")

display(corr.round(2))

In [ ]:
# reconciliar la correlación densidad–SESNSP (+0.72 en el día 4)
#          y la convención del índice de condición (10.9 con 8 candidatas)

a, b = "z_log_densidad", "z_sesnsp_nucleo"

# --- 1. La correlación en distintas bases -------------------------------------
print("Correlación densidad–SESNSP según la base de cálculo:")
print(f"  80 filas alcaldía-año           {covs[a].corr(covs[b]):+.3f}")
prom = covs.groupby("cve_mun")[[a, b]].mean()
print(f"  16 alcaldías, promedio de años  {prom[a].corr(prom[b]):+.3f}")
for anio, g in covs.groupby("anio_ref"):
    print(f"  16 alcaldías, año {anio}          {g[a].corr(g[b]):+.3f}")

# --- 2. Índice de condición: tres convenciones --------------------------------
# (i)  Belsley-Kuh-Welsch: se agrega la columna de unos, cada columna se
#      escala a norma 1 (sin centrar) y se toma el cociente de valores
#      singulares máximo/mínimo. Es la definición estándar en diagnóstico
#      de colinealidad y la que usan olsrr y la mayoría de los textos.
# (ii) Raíz del cociente de eigenvalores de la matriz de correlaciones
#      (variables centradas: ignora la colinealidad con el intercepto).
# (iii) Cociente de eigenvalores sin raíz (el "número de condición" kappa).
def indice_condicion(X, metodo="belsley"):
    X = np.asarray(X, dtype=float)
    if metodo == "belsley":
        Z = np.column_stack([np.ones(len(X)), X])
        Z = Z / np.linalg.norm(Z, axis=0)
        s = np.linalg.svd(Z, compute_uv=False)
        return s.max() / s.min()
    lam = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False))
    return np.sqrt(lam.max() / lam.min()) if metodo == "corr_raiz" \
        else lam.max() / lam.min()

ORIG8 = COV_FIJAS + ["z_sesnsp_nucleo"]   # las 8 candidatas del día 4
print("\nÍndice de condición de las 8 candidatas originales (esperado 10.9):")
for base_nom, X in [("80 filas", covs[ORIG8]),
                    ("16 promedios", covs.groupby("cve_mun")[ORIG8].mean())]:
    for m in ["belsley", "corr_raiz", "kappa"]:
        print(f"  {base_nom:13s} {m:10s} {indice_condicion(X, m):7.1f}")

Decisión sobre el par densidad–SESNSP: aunque en la base de 80 filas queda en 0.61, lo mantengo como par prohibido. Hay dos razones. La especificación por año del día 10 incluye un modelo para 2020, donde la correlación es 0.72. Y conviene que la regla quede idéntica a la documentada en el día 4. 

In [ ]:
# selección por todos los subconjuntos (réplica de olsrr::ols_step_best_subset)

# Diseño de la regresión de selección:
#   - Variable dependiente: theta (escala arcoseno), la escala del Fay-Herriot.
#   - Filas: las 80 celdas alcaldía-año de ocurrencia con directo disponible
#     (la celda "sin estimación" de k no entra, igual que en el modelo).
#   - Efectos fijos de año SIEMPRE incluidos: la especificación agrupada del
#     día 10 los lleva, así que las covariables compiten por explicar lo que
#     los años no explican.
#   - Como máximo 4 covariables (16 áreas × 5 años no dan para más).
#   - Ningún subconjunto puede contener los dos miembros de un par con |r| > 0.70.
#
# Los tres criterios:
#   - R² ajustado: penaliza poco el número de variables; tiende a preferir
#     modelos grandes. Por eso el tope de 4 importa.
#   - AIC = -2·log-verosimilitud + 2·k. Penaliza 2 unidades por parámetro.
#   - Cp de Mallows = SSE_p / σ²_completo − n + 2k, donde σ² se estima con el
#     modelo que incluye TODAS las candidatas. Estima el error de predicción
#     total del subconjunto. Un buen modelo tiene Cp chico y cercano a k:
#     si Cp >> k, al subconjunto le falta algo importante (sesgo).

PARES_PROHIBIDOS = [("z_prop_fem_18", "z_eje_socio_1"),
                    ("z_dist_mp_km", "z_sesnsp_nucleo"),
                    ("z_prop_ocupada_12", "z_agencias_mp_100k"),
                    ("z_eje_socio_1", "z_dist_mp_km"),
                    ("z_log_densidad", "z_sesnsp_nucleo")]   # 0.72 en 2020
prohibidos = [set(p) for p in PARES_PROHIBIDOS]

def permitido(conj):
    return not any(p <= set(conj) for p in prohibidos)

SUBCONJUNTOS = [s for k in range(0, 5)
                for s in itertools.combinations(CANDIDATAS, k) if permitido(s)]
print(f"Subconjuntos de hasta 4 covariables sin pares prohibidos: {len(SUBCONJUNTOS)}")

def seleccion_todos(param):
    df = base[(base["param"] == param) & base["theta"].notna()].reset_index(drop=True)
    y, n = df["theta"].to_numpy(), len(df)
    efectos_anio = pd.get_dummies(df["anio_ref"], prefix="a",
                                  drop_first=True, dtype=float)

    def matriz(conj):
        return sm.add_constant(pd.concat([efectos_anio, df[list(conj)]], axis=1),
                               has_constant="add")

    # σ² de referencia para Cp: modelo con todas las candidatas
    s2_completo = sm.OLS(y, matriz(CANDIDATAS)).fit().mse_resid

    filas = []
    for conj in SUBCONJUNTOS:
        m = sm.OLS(y, matriz(conj)).fit()
        k = int(m.df_model) + 1                       # parámetros, con intercepto
        filas.append({"param": param, "n": n, "n_cov": len(conj),
                      "covariables": ", ".join(c.removeprefix("z_") for c in conj)
                                     or "(solo efectos de año)",
                      "k": k, "cp": m.ssr / s2_completo - n + 2 * k,
                      "aic": m.aic, "r2_aj": m.rsquared_adj, "conj": conj})
    return pd.DataFrame(filas)

sel = pd.concat([seleccion_todos(p) for p in ["d", "k", "R"]], ignore_index=True)

# --- Resumen: los 5 mejores por criterio, para cada parámetro ------------------
cols = ["n_cov", "covariables", "k", "cp", "aic", "r2_aj"]
for p in ["d", "k", "R"]:
    s = sel[sel["param"] == p]
    base_anios = s[s["n_cov"] == 0].iloc[0]
    print(f"\n{'=' * 100}\nParámetro {p}  (n = {s['n'].iloc[0]}; "
          f"R² ajustado solo con efectos de año = {base_anios['r2_aj']:.3f})")
    for crit, asc in [("cp", True), ("aic", True), ("r2_aj", False)]:
        print(f"\n  Mejores 5 por {crit}:")
        print(s.sort_values(crit, ascending=asc).head(5)[cols]
               .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Las covariables casi no explican nada. Los R² ajustados están todos entre −0.05 y +0.03. Las diferencias de AIC entre los mejores modelos son menores a 2 unidades, que por la regla usual significa que los modelos no se distinguen entre sí. Y en k y en R, el modelo con solo efectos de año empata con cualquier conjunto de covariables, o incluso les gana.

**Lectura 1**: el ruido muestral tapa la señal. La variable dependiente es el estimador directo, que contiene dos cosas: la variación verdadera entre alcaldías (la que las covariables pueden explicar) y el error de muestreo (que nada puede explicar). Si el error domina, el R² queda acotado cerca de cero aunque las covariables sean buenas predictoras de la proporción verdadera. Para R, con 51 de 80 celdas por encima de 30% de CV, esto es casi seguro lo que pasa. Es un defecto conocido de aplicar los criterios clásicos de olsrr a estimadores directos. Hasta donde recuerdo, Lahiri y Suntornchost (2015, Sankhya B) propusieron versiones ajustadas del R² y del AIC que descuentan la varianza muestral conocida ψ_d. 

**Lectura 2**: la variación verdadera existe, pero estas covariables no la capturan. Para k esta lectura es la más interesante. k está bien medido (solo 9 celdas con CV alto), así que el ruido no alcanza para explicar un R² de cero. Si k varía de verdad entre alcaldías y ninguna covariable territorial lo explica, ni siquiera la distancia al ministerio público o las agencias por habitante, eso ya es un hallazgo sobre la barrera institucional.

Las dos lecturas se separan con un solo cálculo: la varianza del efecto aleatorio, σ²_u. La varianza total de θ alrededor de la regresión es aproximadamente σ²_u + ψ_d. Como ψ_d es conocida, lo que sobra es la variación verdadera entre áreas. Hay tres casos:

- σ²_u cercana a cero: casi no hay nada que explicar. Además, es exactamente el riesgo de frontera que anotaste para el día 10, solo que aquí lo detectamos un día antes.
- σ²_u grande que no baja al agregar covariables: variación verdadera no explicada (lectura 2).
- σ²_u que baja claramente con un conjunto: ese conjunto sí explica la señal, aunque el R² clásico no lo vea.

De ahí sale un mejor criterio de selección, el R² de la señal: 1 − σ²_u(conjunto) / σ²_u(solo años), la fracción de la variación verdadera entre áreas que explican las covariables.

## Modelo Fay-Herriot

In [ ]:
# ¿cuánta variación verdadera hay entre áreas, y cuánta explican
#          las covariables? Estimador de momentos de Prasad-Rao para σ²_u

# Modelo Fay-Herriot:  θ_d = x_d'β + u_d + e_d,  u_d ~ N(0, σ²_u),  e_d ~ N(0, ψ_d)
#
# Si ajustamos MCO de θ sobre X, los residuales contienen u_d + e_d. Su suma de
# cuadrados tiene esperanza:
#       E[SSE] = (n − p)·σ²_u + Σ ψ_d (1 − h_dd)
# donde h_dd es el apalancamiento de la celda d (diagonal de la matriz sombrero).
# Despejando:
#       σ̂²_u = [SSE − Σ ψ_d (1 − h_dd)] / (n − p)
#
# Es el estimador de momentos de Prasad y Rao (1990). Sirve hoy como diagnóstico
# rápido porque no requiere iterar. Mañana... perdón, en el día 10, la REML lo
# sustituye. Si sale NEGATIVO, la dispersión observada es menor de lo que el
# puro error muestral produciría: no hay señal detectable entre áreas.

def sigma2u_momentos(y, X, psi):
    X = np.asarray(X, dtype=float)
    n, p = X.shape
    Q, _ = np.linalg.qr(X)                   # X = QR; la matriz sombrero es QQ'
    h = (Q ** 2).sum(axis=1)                 # diagonal de la matriz sombrero
    resid = y - Q @ (Q.T @ y)
    return (resid @ resid - np.sum(psi * (1 - h))) / (n - p)

def senal_todos(param):
    df = base[(base["param"] == param) & base["theta"].notna()].reset_index(drop=True)
    y, psi = df["theta"].to_numpy(), df["var_theta_asen"].to_numpy()
    efectos_anio = pd.get_dummies(df["anio_ref"], prefix="a",
                                  drop_first=True, dtype=float)
    matriz = lambda conj: sm.add_constant(
        pd.concat([efectos_anio, df[list(conj)]], axis=1), has_constant="add")

    s2u_anios = sigma2u_momentos(y, matriz(()), psi)
    filas = []
    for conj in SUBCONJUNTOS:
        s2u = sigma2u_momentos(y, matriz(conj), psi)
        filas.append({"param": param, "n_cov": len(conj),
                      "covariables": ", ".join(c.removeprefix("z_") for c in conj)
                                     or "(solo efectos de año)",
                      "s2u": s2u,
                      "r2_senal": 1 - s2u / s2u_anios if s2u_anios > 0 else np.nan,
                      "conj": conj})
    resumen = {"param": param, "var_theta_resid_anios":
               np.var(y - matriz(()) @ np.linalg.lstsq(matriz(()), y, rcond=None)[0], ddof=5),
               "psi_media": psi.mean(), "s2u_anios": s2u_anios}
    return pd.DataFrame(filas), resumen

res_senal, resumenes = zip(*[senal_todos(p) for p in ["d", "k", "R"]])
senal = pd.concat(res_senal, ignore_index=True)

# --- 1. Diagnóstico de señal contra ruido, modelo con solo efectos de año ------
diag = pd.DataFrame(resumenes).set_index("param")
diag["gamma_medio_aprox"] = diag["s2u_anios"].clip(lower=0) / (
    diag["s2u_anios"].clip(lower=0) + diag["psi_media"])
print("Señal contra ruido (modelo solo con efectos de año):")
print(diag.to_string(float_format=lambda x: f"{x:.5f}"))
print("\n  var_theta_resid_anios ≈ σ²_u + ψ medio. Si ψ medio se lleva casi todo,")
print("  la variación observada es casi pura muestra. gamma_medio_aprox es el peso")
print("  típico que tendría el directo en el EBLUP.\n")

# --- 2. Los 5 subconjuntos que más reducen σ²_u, por parámetro ----------------
for p in ["d", "k", "R"]:
    s = senal[senal["param"] == p].sort_values("s2u").head(5)
    print(f"{'=' * 90}\nParámetro {p}: subconjuntos con menor σ²_u")
    print(s[["n_cov", "covariables", "s2u", "r2_senal"]]
          .to_string(index=False, float_format=lambda x: f"{x:.5f}"))

1. La señal existe en los tres parámetros. σ²_u es positiva en todos, así que el escenario de frontera no aparece en el modelo agrupado. (En los modelos por año, con 16 áreas, todavía puede aparecer; lo vigilamos el día 10). Pero el ruido domina: entre 60% y 73% de la dispersión observada es error de muestreo. De ahí los γ típicos de 0.27 a 0.39: en el EBLUP, el modelo va a pesar más que la encuesta en la mayoría de las celdas. Eso es justo lo que justifica usar estimación en áreas pequeñas.

2. El ψ medio de k (0.023) es siete veces el de d y el de R. k es una proporción alta, con base en los denunciantes, que son pocos. El CV divide el error estándar entre la estimación, y una proporción cercana a 1 tiene un CV pequeño aunque su error absoluto sea grande. En escala arcoseno, que no tiene ese sesgo, k resulta ser el parámetro peor medido. Esto va a la nota del día y afecta la tabla de confiabilidad: los umbrales de CV favorecen artificialmente a k.

3. Lo que explican las covariables, ahora medido sobre la señal:

- d: el conjunto {sesnsp_nucleo, prop_7d_std, prop_digital_robo} explica 17% de la variación verdadera. Gana con los cuatro criterios (Cp, AIC, R² ajustado y R² de la señal). La elección es clara.
- R: gana el mismo conjunto, pero solo con 7%. Por AIC y Cp prácticamente empata con usar solo prop_digital_robo o solo los efectos de año.
- k: nada pasa de 2.5%. Ninguna covariable territorial explica la barrera institucional, ni la distancia al ministerio público ni las agencias por habitante. Con 16 áreas, esto no prueba que no haya relación, pero sí que no se detecta. Es un hallazgo: la variación de k entre alcaldías no sigue la geografía del acceso físico, lo que apunta a prácticas de cada sede más que a infraestructura.

Una precaución: el R² de la señal también es ruidoso, y recorrer 230 subconjuntos le da cierto margen al azar. Por eso me fijo en lo que se repite (prop_digital_robo está en todos los finalistas de d y de R) más que en el primer lugar exacto.

**La decisión de circularidad**

No hay circularidad en la estimación: R sale de la encuesta, que es independiente del registro. Pero hay un problema distinto: la dependencia funcional en el mapa corregido.

El EBLUP es R̂ = γ·directo + (1−γ)·(x'β). Con γ cerca de 0.36, en promedio casi dos tercios de R̂ vienen de la parte sintética. Si x incluye sesnsp_nucleo, que mide prácticamente lo mismo que C (el SESNSP compila los datos de la propia Fiscalía), entonces en las celdas con poco peso de la encuesta queda R̂ ≈ a + b·C. El mapa corregido se vuelve C / (a + b·C), una función que comprime las diferencias entre alcaldías por construcción, con una intensidad que depende de la pendiente b. Parte de lo que mostraría el mapa sería un artefacto del modelo, no información de la encuesta.

**Mi propuesta**:

- Especificación principal de R para el mapa: sin SESNSP. Es la que no se puede atacar.
- Con SESNSP como sensibilidad. Si el mapa casi no cambia, el punto queda cerrado. Si cambia, la diferencia se reporta como el tamaño del efecto de compresión.

El costo de esta decisión depende de cuánta señal se pierde al quitar el SESNSP, y eso es lo que calcula la celda siguiente. Para d no hay problema en conservarlo, porque d no entra en el cociente.

In [ ]:
# comparar los conjuntos finalistas y guardar la tabla de selección

FINALISTAS = {
    "d": [("z_sesnsp_nucleo", "z_prop_7d_std", "z_prop_digital_robo"),
          ("z_prop_7d_std", "z_prop_digital_robo")],
    "k": [(),
          ("z_log_densidad", "z_dist_mp_km")],
    "R": [("z_sesnsp_nucleo", "z_prop_7d_std", "z_prop_digital_robo"),
          ("z_prop_7d_std", "z_prop_digital_robo"),          # sin SESNSP
          ("z_prop_digital_robo",)],
}

filas_fin = []
for p, conjuntos in FINALISTAS.items():
    df = base[(base["param"] == p) & base["theta"].notna()].reset_index(drop=True)
    y, psi = df["theta"].to_numpy(), df["var_theta_asen"].to_numpy()
    efectos_anio = pd.get_dummies(df["anio_ref"], prefix="a",
                                  drop_first=True, dtype=float)
    for conj in conjuntos:
        X = sm.add_constant(pd.concat([efectos_anio, df[list(conj)]], axis=1),
                            has_constant="add")
        m = sm.OLS(y, X).fit()
        s2u = sigma2u_momentos(y, X, psi)
        etiqueta = ", ".join(c.removeprefix("z_") for c in conj) or "(solo años)"
        print(f"\n{p} | {etiqueta}")
        print(f"   σ²_u = {s2u:.5f}   R² señal = {1 - s2u / diag.loc[p, 's2u_anios']:.3f}"
              f"   AIC = {m.aic:.2f}")
        # Índice de condición (Belsley): solo covariables, comparable con el 10.9,
        # y con los efectos de año incluidos, que es la matriz real del modelo
        if conj:
            ic_cov = indice_condicion(df[list(conj)])
            ic_tot = indice_condicion(pd.concat([efectos_anio, df[list(conj)]], axis=1))
            print(f"   Índice de condición: covariables {ic_cov:.1f}, con años {ic_tot:.1f}")
            for c in conj:
                print(f"   {c:22s} β = {m.params[c]:+.4f}   t = {m.tvalues[c]:+.2f}")
        filas_fin.append({"param": p, "covariables": etiqueta, "s2u": s2u,
                          "aic": m.aic,
                          "ic_cov": ic_cov if conj else np.nan,
                          "ic_con_anios": ic_tot if conj else np.nan})

finalistas = pd.DataFrame(filas_fin)

# --- Entregable 1: tabla completa de selección (los 230 × 3 subconjuntos) -----
tabla_sel = (sel.drop(columns="conj")
             .merge(senal[["param", "covariables", "s2u", "r2_senal"]],
                    on=["param", "covariables"], validate="1:1"))
tabla_sel.to_csv(RUTA_SALIDA / "seleccion_covariables_todos.csv",
                 index=False, encoding="utf-8-sig")
finalistas.to_csv(RUTA_SALIDA / "seleccion_covariables_finalistas.csv",
                  index=False, encoding="utf-8-sig")
print(f"\nGuardado en {RUTA_SALIDA}")

d: el conjunto de tres se confirma, y el signo es el esperado. prop_digital_robo sale positivo y es el coeficiente más fuerte (t = +2.82): donde hay más canal digital, se denuncia más. El SESNSP sale negativo (t = −2.12): en las alcaldías con más delito registrado, la propensión a denunciar es menor. Es coherente con la literatura sobre desgaste y normalización en zonas de alta victimización.

Además, al quitar el SESNSP, prop_digital_robo baja de t = 2.82 a 1.99 y prop_7d_std se desvanece. El SESNSP funciona como variable supresora: correlaciona positivamente con las otras dos (+0.35 y +0.23), pero su efecto sobre d va en sentido contrario, así que al controlarlo se limpia la relación de las otras. Por eso el conjunto funciona en bloque, y por eso el R² de la señal cae de 0.168 a 0.053 sin él. El índice de condición es 1.9 (5.1 con los años), muy por debajo de 30.

R: la decisión de circularidad. Sin SESNSP, el mejor conjunto es prop_digital_robo sola (R² de señal 0.030). De hecho tiene el mejor AIC de todos los modelos de R (−190.71). Que {7d, digital} quede por debajo de digital sola es el estimador de momentos castigando un grado de libertad que no aporta. Lo que importa es el costo en el EBLUP: con ψ medio de 0.00324, el γ típico pasa de 0.347 a 0.356. Una diferencia de un punto. Quitar el SESNSP prácticamente no cambia cuánto pesa la encuesta.

Hay además un dato que refuerza la decisión: el SESNSP entra a R con signo negativo. El mapa corregido sería entonces C / (a − b·C), que exagera las diferencias en lugar de comprimirlas. El sentido del artefacto es otro, pero el problema es el mismo: parte del mapa saldría del modelo y no de la encuesta.

k: solo efectos de año. La densidad y la distancia al ministerio público salen negativas (t ≈ −1.6), en la dirección que uno esperaría, pero no significativas, y el AIC y el Cp prefieren el modelo sin covariables. El sintético de k será la media anual de la ciudad. Es un **Fay-Herriot legítimo**: encoge hacia la media en vez de hacia una recta. Los coeficientes de acceso se reportan como evidencia del hallazgo, no como modelo.

In [ ]:
# especificación final de covariables 

# Todo lo que sigue (EBLUP agrupado, por año, figuras) lee de este diccionario.
# Cambiar una especificación es cambiar solo aquí.

ESPECIFICACIONES = {
    # (parámetro, etiqueta): covariables
    ("d", "principal"):    ["z_sesnsp_nucleo", "z_prop_7d_std", "z_prop_digital_robo"],
    ("k", "principal"):    [],
    ("k", "sens_acceso"):  ["z_log_densidad", "z_dist_mp_km"],
    ("R", "principal"):    ["z_prop_digital_robo"],
    ("R", "sens_sesnsp"):  ["z_sesnsp_nucleo", "z_prop_7d_std", "z_prop_digital_robo"],
}

# Tabla resumen de la selección (entregable 1)
filas = []
for (p, etq), conj in ESPECIFICACIONES.items():
    fila = finalistas[(finalistas["param"] == p) &
                      (finalistas["covariables"] ==
                       (", ".join(c.removeprefix("z_") for c in conj) or "(solo años)"))]
    filas.append({"param": p, "especificacion": etq,
                  "covariables": ", ".join(conj) or "(solo efectos de año)",
                  "s2u_momentos": fila["s2u"].iloc[0],
                  "r2_senal": 1 - fila["s2u"].iloc[0] / diag.loc[p, "s2u_anios"],
                  "aic": fila["aic"].iloc[0],
                  "ic_cov": fila["ic_cov"].iloc[0],
                  "ic_con_anios": fila["ic_con_anios"].iloc[0]})

seleccion_final = pd.DataFrame(filas)
seleccion_final.to_csv(RUTA_SALIDA / "seleccion_final.csv",
                       index=False, encoding="utf-8-sig")
display(seleccion_final.round(4))

Los índices de condición van de 1.0 a 5.1 contra el tope de 30, y las cinco especificaciones están congeladas. 

### Qué hace la máxima verosimilitud restringida y por qué

El problema de la máxima verosimilitud ordinaria. Pensemos en la varianza muestral: si divides la suma de cuadrados entre n en vez de n − 1, subestimamos la varianza. Los datos están más cerca de su media estimada que de la verdadera, porque la estimación se ajustó a esos mismos datos y “se comió” un grado de libertad.

La máxima verosimilitud tiene el mismo defecto, pero multiplicado. Estima β y σ²_u al mismo tiempo y trata a β̂ como si fuera el valor verdadero. Los residuales alrededor de la recta ajustada son sistemáticamente más chicos que alrededor de la recta verdadera, porque el ajuste absorbió p grados de libertad. Resultado: σ²_u sale sesgada hacia abajo.

Con 80 celdas y 6 parámetros el sesgo es moderado. Con 16 alcaldías y 5 parámetros, que es el modelo por año con covariables, es grave. Y empuja justo hacia el problema que me preocupa: σ²_u igual a cero.

En lugar de la verosimilitud de θ, se usa la verosimilitud de n − p combinaciones lineales de los datos construidas para que no dependan de β, como los residuales. Como su distribución no involucra β, no hay que estimarlo para estimar σ²_u, y el sesgo desaparece. En la fórmula, la diferencia es un solo término:

ℓ_REML(σ²_u) = −½ [ log|V| + log|X'V⁻¹X| + θ'Pθ ]

El término log|X'V⁻¹X| es la corrección: el “costo” de no conocer β. Es el análogo exacto de pasar de n a n − p. Esta es la frase que puedes usar ante el jurado: la REML es a la ML lo que dividir entre n − p es a dividir entre n.

Cómo se maximiza. Con **scoring de Fisher**, que es lo que hace eblupFH. Es un método de Newton donde la curvatura se reemplaza por su valor esperado (la información de Fisher), y eso lo vuelve más estable. En cada paso: σ²_nueva = σ² + score / información.

El caso frontera y la alternativa. Si la derivada de la verosimilitud restringida en σ²_u = 0 ya es negativa, la verosimilitud baja desde el cero y el máximo está en la frontera: γ = 0 en todas las celdas, y el EBLUP se vuelve puramente sintético. 

La alternativa es la **verosimilitud ajustada de Li y Lahiri (2010)**: se multiplica la verosimilitud restringida por σ²_u. En logaritmos, eso suma log σ²_u, que tiende a menos infinito en cero. El máximo queda siempre en el interior y la varianza estimada sale estrictamente positiva. El costo es un sesgo pequeño hacia arriba, que desaparece cuando crece el número de áreas.

Error cuadrático medio. Se descompone en tres partes:

g1: el error del predictor si conociéramos todos los parámetros.
g2: el costo de estimar β.
g3: el costo de estimar σ²_u.

Con la REML, el estimador correcto es g1 + g2 + 2·g3 (Datta y Lahiri, 2000). Para la versión ajustada uso la misma fórmula como aproximación. 

In [ ]:
# Fay-Herriot por REML (y REML ajustada), EBLUP y ECM de Prasad-Rao

def _componentes_reml(s2, y, X, psi):
    """Score, información de Fisher y log-verosimilitud restringida en σ²_u = s2."""
    v = s2 + psi                                  # varianza total de cada celda
    Xv = X / v[:, None]                           # V⁻¹X (V es diagonal)
    A = np.linalg.inv(X.T @ Xv)                   # (X'V⁻¹X)⁻¹
    P = np.diag(1 / v) - Xv @ A @ Xv.T            # proyecta fuera de las columnas de X
    Py = P @ y
    score = -0.5 * np.trace(P) + 0.5 * Py @ Py    # derivada de ℓ_REML
    info = 0.5 * np.sum(P * P)                    # ½ tr(PP), información esperada
    _, logdet_xvx = np.linalg.slogdet(X.T @ Xv)   # el término que hace "restringida"
    loglik = -0.5 * (np.log(v).sum() + logdet_xvx + y @ Py)
    return score, info, loglik

def estimar_s2u(y, X, psi, metodo="reml", tol=1e-10, max_iter=200):
    """Scoring de Fisher. metodo: 'reml' o 'ajustada' (Li-Lahiri: + log σ²_u).
    Devuelve (σ²_u, iteraciones, frontera)."""
    if metodo == "reml":
        score0, _, _ = _componentes_reml(0.0, y, X, psi)
        if score0 <= 0:          # la verosimilitud ya baja desde cero:
            return 0.0, 0, True  # el máximo está en la frontera
    s2 = np.median(psi)          # valor inicial, el mismo que usa eblupFH
    for it in range(1, max_iter + 1):
        score, info, _ = _componentes_reml(s2, y, X, psi)
        if metodo == "ajustada":             # derivadas del término log σ²_u
            score += 1 / s2
            info += 1 / s2 ** 2
        paso = score / info
        while s2 + paso <= 0:                # nunca salir del dominio σ²_u > 0
            paso /= 2
        if abs(paso) < tol:
            return s2 + paso, it, False
        s2 += paso
    print("  AVISO: el scoring de Fisher no convergió")
    return s2, max_iter, False

def ajustar_fh(df, covariables, efectos_anio=True, metodo="reml"):
    """Ajusta un Fay-Herriot sobre las filas de df. Las celdas sin directo
    (theta nulo) reciben la predicción sintética."""
    df = df.reset_index(drop=True)
    partes = [df[covariables]]
    if efectos_anio:
        partes.insert(0, pd.get_dummies(df["anio_ref"], prefix="a",
                                        drop_first=True, dtype=float))
    X_todas = sm.add_constant(pd.concat(partes, axis=1),
                              has_constant="add").to_numpy(float)
    obs = df["theta"].notna().to_numpy()
    theta = df["theta"].to_numpy()
    psi_todas = df["var_theta_asen"].to_numpy()
    y, psi, X = theta[obs], psi_todas[obs], X_todas[obs]

    # 1. Varianza del efecto aleatorio
    s2, iters, frontera = estimar_s2u(y, X, psi, metodo)

    # 2. β por mínimos cuadrados generalizados y predicción sintética x'β
    v = s2 + psi
    Xv = X / v[:, None]
    A = np.linalg.inv(X.T @ Xv)
    beta = A @ (Xv.T @ y)
    sintetico = X_todas @ beta

    # 3. EBLUP: combinación convexa con γ = σ²_u / (σ²_u + ψ)
    gamma = np.where(obs, s2 / (s2 + psi_todas), 0.0)
    eblup = np.where(obs, gamma * np.nan_to_num(theta) + (1 - gamma) * sintetico,
                     sintetico)

    # 4. ECM: g1 + g2 + 2·g3 (REML); celdas sin directo: σ²_u + x'Ax
    xAx = np.einsum("ij,jk,ik->i", X_todas, A, X_todas)
    var_s2 = 2 / np.sum(1 / v ** 2)                     # varianza asintótica de σ̂²_u
    with np.errstate(invalid="ignore"):
        g1 = gamma * psi_todas
        g2 = (1 - gamma) ** 2 * xAx
        g3 = psi_todas ** 2 / (s2 + psi_todas) ** 3 * var_s2
    ecm = np.where(obs, g1 + g2 + 2 * g3, s2 + xAx)

    salida = df[["alc", "nombre", "anio_ref", "param", "tipo_celda"]].assign(
        theta_dir=theta, psi=psi_todas, sintetico=sintetico, gamma=gamma,
        theta_eblup=eblup, ecm_theta=ecm)
    info = {"s2u": s2, "iteraciones": iters, "frontera": frontera,
            "beta": beta, "n": int(obs.sum()), "p": X.shape[1]}
    return salida, info

# =============================================================================
# Prueba: d, especificación principal, agrupada con efectos de año
# =============================================================================
datos_d = base[base["param"] == "d"]
covs_d = ESPECIFICACIONES[("d", "principal")]

for metodo in ["reml", "ajustada"]:
    res, info = ajustar_fh(datos_d, covs_d, metodo=metodo)
    print(f"{metodo:9s} σ²_u = {info['s2u']:.6f}  iteraciones = {info['iteraciones']}"
          f"  frontera = {info['frontera']}")
print(f"momentos  σ²_u = {diag.loc['d', 's2u_anios']:.6f} (solo años) | "
      f"{finalistas.query('param == \"d\"')['s2u'].iloc[0]:.6f} (con covariables)")

# --- Verificación 1: el máximo de REML coincide con el de una malla ------------
res, info = ajustar_fh(datos_d, covs_d, metodo="reml")
df_obs = datos_d[datos_d["theta"].notna()].reset_index(drop=True)
X_obs = sm.add_constant(pd.concat(
    [pd.get_dummies(df_obs["anio_ref"], prefix="a", drop_first=True, dtype=float),
     df_obs[covs_d]], axis=1), has_constant="add").to_numpy(float)
malla = np.linspace(1e-6, 0.01, 2000)
ll = [_componentes_reml(s, df_obs["theta"].to_numpy(), X_obs,
                        df_obs["var_theta_asen"].to_numpy())[2] for s in malla]
print(f"\nMáximo en la malla: σ²_u = {malla[np.argmax(ll)]:.6f} "
      f"(scoring: {info['s2u']:.6f})")

# --- Verificación 2: el EBLUP queda entre el directo y el sintético -----------
o = res["theta_dir"].notna()
lo = np.minimum(res.loc[o, "theta_dir"], res.loc[o, "sintetico"])
hi = np.maximum(res.loc[o, "theta_dir"], res.loc[o, "sintetico"])
entre = ((res.loc[o, "theta_eblup"] >= lo - 1e-12) &
         (res.loc[o, "theta_eblup"] <= hi + 1e-12)).all()
print(f"EBLUP entre directo y sintético en todas las celdas: {entre}")

# --- Verificación 3: γ baja donde ψ sube --------------------------------------
rho = res.loc[o, ["gamma", "psi"]].corr(method="spearman").iloc[0, 1]
print(f"Correlación de Spearman entre γ y ψ: {rho:+.3f} (debe ser −1)")
print(f"γ: mínimo {res.loc[o, 'gamma'].min():.3f}, mediana "
      f"{res.loc[o, 'gamma'].median():.3f}, máximo {res.loc[o, 'gamma'].max():.3f}")

# --- Verificación 4: el ECM del EBLUP es menor que ψ --------------------------
print(f"Celdas con ECM < ψ: {(res.loc[o, 'ecm_theta'] < res.loc[o, 'psi']).sum()} de {o.sum()}")

El scoring coincide con la malla (0.000289 contra 0.000291), el EBLUP queda acotado, γ y ψ tienen correlación de Spearman −1, y el ECM baja en las 80 celdas.

Pero hay una discrepancia: la REML da σ²_u = 0.00029, y el estimador de momentos daba 0.00176, seis veces más. Las consecuencias son serias. La mediana de γ cae a 0.11, contra 0.39, así que el EBLUP de d sería casi puramente sintético.

La discrepancia viene de los datos o de los supuestos. Los dos estimadores ponderan distinto:

- Momentos suma los residuales al cuadrado sin ponderar: cada celda cuenta igual.
- REML pondera cada celda por 1/(σ²_u + ψ)²: las celdas con muestra grande dominan.

Si unas pocas celdas con muestra chica tienen residuales enormes, inflan el estimador de momentos y casi no mueven la REML. Tengo un candidato concreto: la celda degenerada de d, con θ = 0 (nadie denunció en esa celda de la muestra). Si el d típico ronda 0.10, θ ronda 0.32, y esa sola celda tiene un residual al cuadrado de alrededor de 0.10. Dividido entre los 72 grados de libertad da unos 0.0014, que es casi exactamente la diferencia entre los dos estimadores (0.00176 − 0.00029 ≈ 0.0015). 

La otra posibilidad es más incómoda: que la ψ suavizada subestime la varianza real de las celdas chicas. En ese caso el exceso no está en una celda, sino repartido en todas las de muestra pequeña. La celda siguiente distingue entre las dos.

In [ ]:
# ¿por qué REML y momentos difieren tanto?


def matriz_X(df, covariables, efectos_anio=True):
    partes = [df[covariables]]
    if efectos_anio:
        partes.insert(0, pd.get_dummies(df["anio_ref"], prefix="a",
                                        drop_first=True, dtype=float))
    return sm.add_constant(pd.concat(partes, axis=1),
                           has_constant="add").to_numpy(float)

# --- 1. Ambos estimadores, con y sin celdas degeneradas -----------------------
filas = []
for p in ["d", "k", "R"]:
    covs_p = ESPECIFICACIONES[(p, "principal")]
    for excluir in [False, True]:
        df = base[(base["param"] == p) & base["theta"].notna()]
        if excluir:
            df = df[df["tipo_celda"] != "degenerada"]
        df = df.reset_index(drop=True)
        y, psi, X = (df["theta"].to_numpy(), df["var_theta_asen"].to_numpy(),
                     matriz_X(df, covs_p))
        s2_reml, _, front = estimar_s2u(y, X, psi, "reml")
        filas.append({"param": p, "sin_degeneradas": excluir, "n": len(df),
                      "s2u_momentos": sigma2u_momentos(y, X, psi),
                      "s2u_reml": s2_reml, "frontera": front})
print("σ²_u por estimador, especificación principal agrupada:")
print(pd.DataFrame(filas).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# --- 2. Residuales estandarizados: ¿quién está lejos del modelo? ---------------
# r = (θ_directo − sintético) / sqrt(σ²_u + ψ). Bajo el modelo, r ~ N(0, 1):
# valores con |r| > 3 son atípicos.
for p in ["d", "k", "R"]:
    res, info = ajustar_fh(base[base["param"] == p],
                           ESPECIFICACIONES[(p, "principal")], metodo="reml")
    res = res[res["theta_dir"].notna()].copy()
    res["r"] = (res["theta_dir"] - res["sintetico"]) / np.sqrt(info["s2u"] + res["psi"])
    res = res.merge(base.loc[base["param"] == p, ["alc", "anio_ref", "n", "est"]],
                    on=["alc", "anio_ref"])
    print(f"\n{p}: σ²_u REML = {info['s2u']:.6f}.  Celdas con |r| más grande:")
    print(res.reindex(res["r"].abs().sort_values(ascending=False).index)
             .head(6)[["nombre", "anio_ref", "tipo_celda", "n", "est",
                       "theta_dir", "sintetico", "psi", "r"]]
             .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # --- 3. ¿La ψ suavizada es adecuada según el tamaño de muestra? -----------
    # En cada tercil de ψ: promedio de (θ − sintético)² − ψ. Si el modelo y ψ
    # son correctos, ese exceso debe ser parecido en los tres terciles (≈ σ²_u).
    # Si crece con ψ, la ψ suavizada se queda corta en las celdas chicas.
    res["tercil_psi"] = pd.qcut(res["psi"], 3, labels=["ψ baja (n grande)",
                                                       "ψ media", "ψ alta (n chica)"])
    exceso = (res.assign(exceso=(res["theta_dir"] - res["sintetico"]) ** 2 - res["psi"])
                 .groupby("tercil_psi", observed=True)["exceso"].agg(["mean", "median", "count"]))
    print(f"  Exceso de dispersión por tercil de ψ ({p}):")
    print(exceso.to_string(float_format=lambda x: f"{x:.5f}"))

El diagnóstico confirma la hipótesis principal y descarta la incómoda.

1. Las celdas degeneradas explican la mayor parte de la discrepancia. Milpa Alta 2022 (d = 0 con 51 casos) aparece como el residual más grande en d y en R. Al quitar las degeneradas, el estimador de momentos cae de 0.00176 a 0.00042 en d y de 0.00179 a 0.00094 en R. La REML se mueve mucho menos (0.00029 → 0.00018 y 0.00047 → 0.00041): pondera por precisión, así que una celda con muestra chica la arrastra poco. La REML es el estimador robusto aquí, y es el que usa eblupFH por defecto.

2. La ψ suavizada no está fallando de forma sistemática. Las medianas del exceso de dispersión son negativas en los tres tercios de los tres parámetros. Eso es lo esperado: una variable tipo ji-cuadrada tiene la mediana por debajo de la media. Los promedios altos del tercil de muestra chica se deben a celdas individuales (Milpa Alta 2022 y las de k con valores extremos), no a un patrón. No hay que tocar nada del día 5.

3. k tiene variación verdadera y robusta. Su REML casi no cambia sin las degeneradas (0.0044 → 0.0040). Iztacalco 2020 con k = 0.97 junto a Azcapotzalco 2020 con k = 0.33, ambos con muestras de 20 casos, ilustran por qué.

La consecuencia: los γ reales son mucho más bajos. Con la REML, el γ típico ronda 0.10 en d, 0.13 en R y 0.16 en k. El EBLUP va a pesar 85–90% el modelo. Dice que la mayor parte de la dispersión entre alcaldías que muestran los directos es ruido de muestreo, y que una vez descontadas las covariables y los años queda poca variación verdadera. El mapa modelado de R va a verse mucho más plano que el de los directos.

La selección se apoyó en el estimador de momentos, que estaba contaminado por las celdas degeneradas. Los criterios clásicos (Cp, AIC) también, porque son de mínimos cuadrados ordinarios. Antes de correr los modelos hay que confirmar que los conjuntos elegidos siguen siendo los correctos con la REML. Son pocos ajustes.

Mi propuesta para las celdas degeneradas: en la especificación principal, mantener la decisión del día 5 (entran al ajuste con su θ y su ψ), por coherencia con lo ya documentado y porque la REML las tolera bien. Como sensibilidad, ajustar sin ellas y predecirlas como sintéticas. Esa sensibilidad va en la nota, junto con la explicación de por qué un θ en el borde viola el supuesto de normalidad que el arcoseno no alcanza a corregir.

In [ ]:
# revisar la selección del día 9 con REML, con y sin degeneradas

# Ahora el R² de la señal se calcula con σ²_u de REML:
#     R²_señal = 1 − σ²_u(conjunto) / σ²_u(solo años)
# Puede salir negativo: con REML, agregar una covariable inútil no garantiza
# que σ²_u baje (los grados de libertad se descuentan).

filas = []
for p, conjs in FINALISTAS.items():
    conjs = [()] + [c for c in conjs if c != ()]          # siempre incluir "solo años"
    for excluir in [False, True]:
        df = base[(base["param"] == p) & base["theta"].notna()]
        if excluir:
            df = df[df["tipo_celda"] != "degenerada"]
        df = df.reset_index(drop=True)
        y, psi = df["theta"].to_numpy(), df["var_theta_asen"].to_numpy()
        s2_anios = estimar_s2u(y, matriz_X(df, []), psi)[0]
        for conj in conjs:
            s2, _, front = estimar_s2u(y, matriz_X(df, list(conj)), psi)
            filas.append({
                "param": p, "sin_degeneradas": excluir,
                "covariables": ", ".join(c.removeprefix("z_") for c in conj) or "(solo años)",
                "s2u_reml": s2, "frontera": front,
                "r2_senal_reml": 1 - s2 / s2_anios if s2_anios > 0 else np.nan,
                "gamma_mediana": np.median(s2 / (s2 + psi))})

revision = pd.DataFrame(filas)
for p in ["d", "k", "R"]:
    print(f"\n{'=' * 90}\nParámetro {p}")
    print(revision[revision["param"] == p]
          .drop(columns="param")
          .to_string(index=False, float_format=lambda x: f"{x:.5f}"))

revision.to_csv(RUTA_SALIDA / "seleccion_revision_reml.csv",
                index=False, encoding="utf-8-sig")

- **d**: el conjunto elegido explica 60% de la señal (67% sin las degeneradas), no el 17% que estimaban los momentos. Las celdas degeneradas inflaban el denominador y tapaban el aporte de las covariables. El conjunto completo sigue muy por encima de la versión sin SESNSP (0.60 contra 0.18), así que la función supresora del SESNSP se confirma.
- **k**: con REML, las covariables de acceso aumentan σ²_u en 11%: no solo no ayudan, sino que estorban. El modelo de solo efectos de año queda confirmado sin ambigüedad, y el hallazgo sobre la barrera institucional se refuerza.
- **R**: prop_digital_robo sola explica 24% y sigue ganándole a {7d, digital} (19%). Todas las clasificaciones son idénticas con y sin degeneradas.

Con los momentos estimé que quitar el SESNSP costaba “un punto de γ”. Con la REML  la señal explicada baja de 43% a 24%, y σ²_u sube de 0.00035 a 0.00047. Sigo recomendando la versión sin SESNSP como principal, porque el argumento de fondo no cambia: ese 19% adicional de poder predictivo viene precisamente de una medida del registro, y es justo lo que daría forma mecánica al cociente C / R̂. 

In [ ]:
# ajuste de todas las especificaciones

# Para cada (parámetro, especificación de covariables):
#   - agrupado: las 80 celdas con efectos fijos de año
#   - por_anio: cinco modelos independientes de 16 alcaldías, sin efectos de año
# Cada uno con REML y con REML ajustada (Li-Lahiri).
# Regla de elección: REML, salvo que caiga en la frontera; entonces, ajustada.
# Además, como sensibilidad: el agrupado principal sin celdas degeneradas.

resultados, resumen = [], []

def correr(df, param, covs, modelo, etiqueta, anio, metodo):
    res, info = ajustar_fh(df, covs, efectos_anio=(modelo == "agrupado"), metodo=metodo)
    o = res["theta_dir"].notna()
    lo = np.minimum(res.loc[o, "theta_dir"], res.loc[o, "sintetico"]) - 1e-12
    hi = np.maximum(res.loc[o, "theta_dir"], res.loc[o, "sintetico"]) + 1e-12
    resultados.append(res.assign(modelo=modelo, especificacion=etiqueta,
                                 ajuste_anio=str(anio), metodo=metodo))
    resumen.append({
        "param": param, "modelo": modelo, "especificacion": etiqueta,
        "ajuste_anio": str(anio), "metodo": metodo,
        "n": info["n"], "p": info["p"], "s2u": info["s2u"],
        "frontera": info["frontera"], "iteraciones": info["iteraciones"],
        "gamma_mediana": res.loc[o, "gamma"].median(),
        "ecm_medio": res.loc[o, "ecm_theta"].mean(),
        "psi_medio": res.loc[o, "psi"].mean(),
        "ok_entre": bool(res.loc[o, "theta_eblup"].between(lo, hi).all()),
        "ok_ecm": bool((res.loc[o, "ecm_theta"] < res.loc[o, "psi"]).all())})

for (p, etq), covs in ESPECIFICACIONES.items():
    datos = base[base["param"] == p]
    for metodo in ["reml", "ajustada"]:
        correr(datos, p, covs, "agrupado", etq, "todos", metodo)
        for anio, g in datos.groupby("anio_ref"):
            correr(g, p, covs, "por_anio", etq, anio, metodo)

# Sensibilidad: degeneradas fuera del ajuste (reciben la predicción sintética)
for p in ["d", "k", "R"]:
    datos = base[base["param"] == p].copy()
    datos.loc[datos["tipo_celda"] == "degenerada", "theta"] = np.nan
    correr(datos, p, ESPECIFICACIONES[(p, "principal")], "agrupado",
           "principal_sin_degeneradas", "todos", "reml")

resumen = pd.DataFrame(resumen)
eblup_theta = pd.concat(resultados, ignore_index=True)

# --- Elección del método por ajuste: REML salvo frontera ----------------------
llave = ["param", "modelo", "especificacion", "ajuste_anio"]
front = (resumen[resumen["metodo"] == "reml"]
         .set_index(llave)["frontera"].rename("frontera_reml"))
resumen = resumen.join(front, on=llave)
resumen["elegido"] = np.where(resumen["frontera_reml"],
                              resumen["metodo"] == "ajustada",
                              resumen["metodo"] == "reml")
eblup_theta = eblup_theta.merge(
    resumen[llave + ["metodo", "elegido"]], on=llave + ["metodo"], validate="m:1")

# --- 1. Verificaciones en todos los ajustes ----------------------------------
print(f"Ajustes realizados: {len(resumen)}")
print(f"  EBLUP fuera del intervalo directo–sintético: {(~resumen['ok_entre']).sum()}")
print(f"  Ajustes con alguna celda ECM ≥ ψ:            {(~resumen['ok_ecm']).sum()}")

# --- 2. Modelos por año: ¿dónde cae la REML en la frontera? -------------------
print("\nσ²_u por REML, modelos por año (* = frontera):")
pa = resumen[(resumen["modelo"] == "por_anio") & (resumen["metodo"] == "reml")].copy()
pa["s2u_txt"] = pa.apply(lambda r: f"{r['s2u']:.5f}" + ("*" if r["frontera"] else " "),
                         axis=1)
print(pa.pivot_table(index=["param", "especificacion"], columns="ajuste_anio",
                     values="s2u_txt", aggfunc="first").to_string())

# --- 3. Agrupado contra por año: ECM medio (método elegido) -------------------
print("\nECM medio en escala θ, método elegido:")
comp = (resumen[resumen["elegido"] & (resumen["especificacion"] != "principal_sin_degeneradas")]
        .groupby(["param", "especificacion", "modelo"])[["ecm_medio", "psi_medio"]]
        .mean().unstack("modelo"))
comp[("razon", "agrupado/por_anio")] = (comp[("ecm_medio", "agrupado")]
                                        / comp[("ecm_medio", "por_anio")])
print(comp.to_string(float_format=lambda x: f"{x:.5f}"))

# Respaldo intermedio, por si algo falla más adelante
eblup_theta.to_parquet(RUTA_SALIDA / "eblup_theta_crudo.parquet", index=False)
resumen.to_csv(RUTA_SALIDA / "eblup_resumen_ajustes.csv", index=False, encoding="utf-8-sig")

El agrupado gana con claridad. Su ECM medio es entre 35% y 61% del de los modelos por año. En d principal, el agrupado reduce la varianza del directo de 0.00335 a 0.00053, una reducción de 84%. 

La frontera aparece en los modelos por año, en 4 de 25 ajustes, y se concentra en 2023 (R, las dos especificaciones de k), con d 2023 también casi en cero (0.0001). Además, la σ²_u por año de d salta entre 0.0001 y 0.0021, un factor de veinte entre años. Esa inestabilidad es exactamente la que justifica el agrupado hoy y el Rao-Yu mañana: con 16 áreas, la varianza del efecto aleatorio no se puede estimar bien año por año.

La que hay que entender: 35 ajustes tienen alguna celda con ECM ≥ ψ. No es un error de programación: el EBLUP quedó dentro del intervalo directo–sintético en los 63 ajustes, y el agrupado de d pasó la verificación completa en la celda 10. Mi hipótesis es que se trata de una propiedad real del ECM en modelos con pocas áreas, que se concentra en las celdas con muestra grande.

El término 2·g3, el costo de estimar σ²_u, es aproximadamente 4ψ̄² / (n·ψ_d), donde ψ̄ es la ψ media del ajuste y ψ_d la de la celda. Para una celda cuya ψ_d es 40% de la media, en un ajuste de 16 áreas, ese término solo ya vale 4ψ̄² / (16 · 0.4ψ̄) ≈ 0.6ψ̄ ≈ 1.6ψ_d. Más que la varianza del directo. Cuando hay pocas áreas, la incertidumbre de haber estimado σ²_u cuesta más de lo que el encogimiento gana, justo en las celdas que ya estaban bien medidas. En esas celdas, el directo es mejor que el EBLUP.

Si la hipótesis es correcta, las fallas deberían estar casi todas en los modelos por año y en las celdas con ψ baja. La celda siguiente lo verifica.

In [ ]:
# Localizar las celdas con ECM ≥ ψ

obs = eblup_theta[eblup_theta["theta_dir"].notna()].copy()
obs["falla"] = obs["ecm_theta"] >= obs["psi"]
obs["razon_ecm_psi"] = obs["ecm_theta"] / obs["psi"]

# --- 1. ¿En qué ajustes? ------------------------------------------------------
print("Celdas con ECM ≥ ψ por tipo de ajuste:")
t = (obs.groupby(["modelo", "metodo", "elegido"])
        .agg(celdas=("falla", "size"), fallas=("falla", "sum")))
t["pct"] = 100 * t["fallas"] / t["celdas"]
print(t.to_string(float_format=lambda x: f"{x:.1f}"))

# --- 2. ¿En qué celdas? Tercil de ψ DENTRO de cada ajuste ---------------------
# Si la hipótesis es correcta, las fallas están en el tercil de ψ baja
# (las celdas con más muestra, donde el directo ya era bueno).
llave_ajuste = ["param", "modelo", "especificacion", "ajuste_anio", "metodo"]
obs["tercil_psi"] = (obs.groupby(llave_ajuste)["psi"]
                        .transform(lambda s: pd.qcut(s.rank(method="first"), 3,
                                                     labels=["baja", "media", "alta"])))
elegidas = obs[obs["elegido"]]
print("\nProporción de fallas por tercil de ψ (solo métodos elegidos):")
print(elegidas.pivot_table(index="modelo", columns="tercil_psi",
                           values="falla", aggfunc="mean", observed=True)
              .to_string(float_format=lambda x: f"{x:.2f}"))

# --- 3. ¿Cuánto se pasan? -----------------------------------------------------
print("\nRazón ECM/ψ en las celdas que fallan (métodos elegidos):")
print(elegidas[elegidas["falla"]].groupby("modelo")["razon_ecm_psi"]
      .describe()[["count", "50%", "max"]].to_string(float_format=lambda x: f"{x:.2f}"))

# --- 4. Especificaciones principales del modelo agrupado -----------------------
print("\nAgrupado, método elegido, fallas por especificación:")
print(elegidas[elegidas["modelo"] == "agrupado"]
      .groupby(["param", "especificacion"])["falla"].agg(["sum", "size"]).to_string())

Hipótesis confirmada:

El agrupado no tiene ninguna falla, en ningún método ni especificación: 633 celdas de 633 con ECM menor que ψ. El resultado principal está limpio.
Las fallas de los modelos por año están todas en el tercil de ψ baja (19% de esas celdas, cero en los otros dos), con una razón mediana de 1.11 y máxima de 1.50. Son celdas con muestra grande donde el directo ya era bueno, y el costo de estimar σ²_u con 16 áreas se come la ganancia.
Mi sospecha sobre la ajustada era incorrecta. La que más falla es la REML no elegida (13.8%). Son los ajustes que cayeron en la frontera: con σ²_u = 0, el término g3 se dispara en las celdas de ψ chica. La ajustada, en cambio, los rescata parcialmente (6.2%).

**Regla adoptada**: en cada celda se reporta el EBLUP si su ECM es menor que ψ, y el directo en caso contrario, con una columna que lo marca. En el agrupado no cambia nada; en los modelos por año afecta 28 celdas. Además, es un argumento más a favor del agrupado.

**El regreso a la escala de proporción**

Por qué no basta con sin²(θ̂). Que θ̂ sea un buen estimador de θ no implica que sin²(θ̂) lo sea de p = sin²(θ): una función no lineal de un estimador insesgado es sesgada. La corrección sale de forma exacta con una identidad trigonométrica. Como sin²x = (1 − cos 2x)/2, y para una normal T ~ N(θ, v) se cumple E[cos 2T] = cos(2θ)·e^{−2v} (es la función característica de la normal), entonces:

E[sin² T] = ½ − ½ · cos(2θ) · e^{−2v}

Se reemplaza θ por el EBLUP y v por su ECM. Para hacerte una idea del tamaño: en d (θ ≈ 0.33, ECM ≈ 0.0005) la corrección mueve la proporción en el cuarto decimal. En los directos, con ψ seis veces mayor, pesa más. Es chica, pero es lo correcto. Y tiene una propiedad útil: nunca produce valores fuera de [0, 1], ni siquiera en las celdas degeneradas.

**Por qué el ECM en proporción necesita bootstrap**. La aproximación lineal (método delta) da ECM_p ≈ sin²(2θ)·ECM_θ. Pero la derivada sin 2θ vale cero en θ = 0 y en θ = π/2: en las celdas degeneradas y en las de k cercanas a 1, el método delta diría que el error es cero. Por eso los intervalos se hacen con un bootstrap paramétrico. Esta celda calcula el método delta como referencia rápida; la siguiente hace el bootstrap.

In [ ]:
#  regla "el modelo solo entra donde mejora" y regreso a proporción=====

def p_desde_theta(theta, var):
    """Media de sin²(T) con T ~ N(theta, var).
    sin²x = (1 − cos 2x)/2  y  E[cos 2T] = cos(2θ)·exp(−2·var)."""
    return 0.5 - 0.5 * np.cos(2 * theta) * np.exp(-2 * var)

eb = eblup_theta.merge(
    base[["alc", "anio_ref", "param", "n", "est", "ee", "cv",
          "conf_inegi", "conf_builgil"]],
    on=["alc", "anio_ref", "param"], how="left", validate="m:1")

# --- 1. Regla de respaldo: directo donde el EBLUP no mejora ----------------
eb["usa_directo"] = eb["theta_dir"].notna() & (eb["ecm_theta"] >= eb["psi"])
eb["theta_final"] = np.where(eb["usa_directo"], eb["theta_dir"], eb["theta_eblup"])
eb["ecm_theta_final"] = np.where(eb["usa_directo"], eb["psi"], eb["ecm_theta"])

# --- 2. Proporción: ingenua y con corrección de sesgo ------------------------
eb["p_ingenuo"] = np.sin(eb["theta_final"].clip(0, np.pi / 2)) ** 2
eb["p_modelo"] = p_desde_theta(eb["theta_final"], eb["ecm_theta_final"])
# Donde se usa el directo, se reporta la proporción directa del día 5, tal cual
eb.loc[eb["usa_directo"], "p_modelo"] = eb.loc[eb["usa_directo"], "est"]

# --- 3. ECM en proporción por método delta (referencia; el bootstrap lo sustituye)
eb["ecm_p_delta"] = np.sin(2 * eb["theta_final"]) ** 2 * eb["ecm_theta_final"]
eb.loc[eb["usa_directo"], "ecm_p_delta"] = eb.loc[eb["usa_directo"], "ee"] ** 2
eb["cv_delta"] = np.sqrt(eb["ecm_p_delta"]) / eb["p_modelo"]

# --- Resumen: especificaciones principales, método elegido -------------------
sel_ = eb["elegido"] & (eb["especificacion"] == "principal")
print("Tamaño de la corrección de sesgo, |p_modelo − p_ingenuo| (EBLUP, no directo):")
print(eb[sel_ & ~eb["usa_directo"]]
      .assign(corr=lambda x: (x["p_modelo"] - x["p_ingenuo"]).abs())
      .groupby(["modelo", "param"])["corr"].agg(["median", "max"])
      .to_string(float_format=lambda x: f"{x:.5f}"))

print("\nCV mediano: directo contra modelado (método delta):")
print(eb[sel_ & eb["theta_dir"].notna()]
      .groupby(["modelo", "param"])[["cv", "cv_delta"]].median()
      .to_string(float_format=lambda x: f"{x:.3f}"))

print("\nCeldas donde la regla de respaldo usa el directo:")
print(eb[eb["elegido"]].groupby(["modelo", "param", "especificacion"])["usa_directo"]
      .sum().loc[lambda s: s > 0].to_string())

# Celdas donde el método delta no sirve (θ en el borde → derivada cero)
borde = sel_ & ((eb["theta_final"] < 0.05) | (eb["theta_final"] > np.pi / 2 - 0.05))
print(f"\nCeldas principales con θ cerca del borde (delta no confiable): {borde.sum()}")

- El CV mediano se reduce a la mitad en el agrupado: R pasa de 0.332 a 0.170, d de 0.278 a 0.128 y k de 0.178 a 0.095. Por año la mejora es menor (0.229, 0.191, 0.121), que es otro argumento a favor del agrupado.
- La corrección de sesgo es pequeña, del orden de milésimas. Es mayor en k y en los modelos por año, que es donde el ECM es más grande, lo cual también cuadra con la fórmula.
- Ninguna celda final queda en el borde. El EBLUP aleja del 0 a las degeneradas: Milpa Alta 2022 ya no estará en d = 0, sino cerca de su sintético. Por eso el método delta resulta razonable aquí. Aun así, el **bootstrap** sigue haciendo falta por dos razones: el método delta es lineal y subestima el error cuando la curvatura importa, y no da intervalos asimétricos, que son los correctos para una proporción.
- La regla de respaldo toca 28 celdas, todas en los modelos por año.

## Bootstrap paramétrico para el modelo agrupado

La idea, siguiendo a González-Manteiga y coautores (2008) para Fay-Herriot (verifica la referencia), es usar el modelo ajustado como si fuera la verdad, generar muchos mundos ficticios a partir de él y ver qué tan lejos queda el estimador de la verdad conocida de cada mundo. En cada réplica:

1. Se genera un efecto u* de cada celda con varianza σ̂²_u. La proporción “verdadera” de esa réplica es p* = sin²(x'β̂ + u*).
2. Se le suma un error muestral e* con varianza ψ_d, lo que da un directo ficticio.
3. Se reajusta el Fay-Herriot completo sobre ese directo ficticio, incluida la estimación de σ²_u. Así el bootstrap capta la incertidumbre de todos los parámetros, no solo la del predictor.
4. Se regresa a proporción y se guarda el error p̂* − p*.

Con mil réplicas, el promedio de los errores al cuadrado es el ECM. Los percentiles de los errores dan el intervalo, que puede ser asimétrico, como debe ser para una proporción.

In [ ]:
#  bootstrap paramétrico (González-Manteiga et al., 2008), agrupado

import time

def ajustar_con_regla(df, covs, efectos_anio=True):
    """REML; si cae en la frontera, REML ajustada. Misma regla que en la celda 13."""
    res, info = ajustar_fh(df, covs, efectos_anio, "reml")
    frontera = info["frontera"]
    if frontera:
        res, info = ajustar_fh(df, covs, efectos_anio, "ajustada")
    return res, info, frontera

def datos_espec(p, etq):
    datos = base[base["param"] == p].copy()
    if etq == "principal_sin_degeneradas":
        datos.loc[datos["tipo_celda"] == "degenerada", "theta"] = np.nan
        covs = ESPECIFICACIONES[(p, "principal")]
    else:
        covs = ESPECIFICACIONES[(p, etq)]
    return datos.reset_index(drop=True), covs

def bootstrap_fh(datos, covs, B, rng):
    res0, info0, _ = ajustar_con_regla(datos, covs)
    mu = matriz_X(datos, covs) @ info0["beta"]          # x'β̂ en todas las celdas
    s2 = info0["s2u"]
    obs = datos["theta"].notna().to_numpy()
    sd_e = np.sqrt(np.nan_to_num(datos["var_theta_asen"].to_numpy()))
    errores = np.empty((B, len(datos)))
    n_frontera = 0
    for b in range(B):
        theta_verdad = mu + rng.normal(0, np.sqrt(s2), len(datos))
        p_verdad = np.sin(np.clip(theta_verdad, 0, np.pi / 2)) ** 2
        d_b = datos.copy()
        d_b["theta"] = np.where(obs, theta_verdad + rng.normal(0, sd_e), np.nan)
        res_b, _, front_b = ajustar_con_regla(d_b, covs)
        n_frontera += front_b
        p_hat = p_desde_theta(res_b["theta_eblup"].to_numpy(),
                              res_b["ecm_theta"].to_numpy())
        errores[b] = p_hat - p_verdad
    p0 = p_desde_theta(res0["theta_eblup"].to_numpy(), res0["ecm_theta"].to_numpy())
    salida = res0[["alc", "anio_ref", "param"]].assign(
        p_boot_ref=p0,
        sesgo_boot=errores.mean(axis=0),
        ecm_p_boot=(errores ** 2).mean(axis=0),
        # Intervalo "básico": p̂ − percentiles del error (corrige asimetría)
        ic95_inf=np.clip(p0 - np.quantile(errores, 0.975, axis=0), 0, 1),
        ic95_sup=np.clip(p0 - np.quantile(errores, 0.025, axis=0), 0, 1))
    return salida, n_frontera

B = 1000
rng = np.random.default_rng(20261012)          # semilla: la fecha de hoy
ESPECS_AGRUPADO = [("d", "principal"), ("d", "principal_sin_degeneradas"),
                   ("k", "principal"), ("k", "sens_acceso"),
                   ("k", "principal_sin_degeneradas"),
                   ("R", "principal"), ("R", "sens_sesnsp"),
                   ("R", "principal_sin_degeneradas")]

boot = []
for p, etq in ESPECS_AGRUPADO:
    t0 = time.time()
    datos, covs = datos_espec(p, etq)
    salida, n_front = bootstrap_fh(datos, covs, B, rng)
    boot.append(salida.assign(especificacion=etq, modelo="agrupado"))
    print(f"{p} {etq:27s} {time.time() - t0:5.1f} s   réplicas en frontera: {n_front}")
boot = pd.concat(boot, ignore_index=True)

# --- Unir al resultado y verificar consistencia --------------------------------
eb = eb.merge(boot, on=["alc", "anio_ref", "param", "modelo", "especificacion"],
              how="left", validate="m:1")
m = eb["elegido"] & (eb["modelo"] == "agrupado")
dif = (eb.loc[m, "p_boot_ref"] - eb.loc[m, "p_modelo"]).abs().max()
print(f"\nVerificación: p del bootstrap vs p_modelo, diferencia máxima {dif:.2e}")

eb["cv_boot"] = np.sqrt(eb["ecm_p_boot"]) / eb["p_modelo"]
print("\nCV mediano, agrupado: directo, delta y bootstrap")
print(eb[m & eb["theta_dir"].notna()]
      .groupby(["param", "especificacion"])[["cv", "cv_delta", "cv_boot"]].median()
      .to_string(float_format=lambda x: f"{x:.3f}"))
print("\nSesgo relativo mediano del bootstrap (sesgo / p):")
print(eb[m].assign(sr=lambda x: x["sesgo_boot"] / x["p_modelo"])
      .groupby(["param", "especificacion"])["sr"].median()
      .to_string(float_format=lambda x: f"{x:+.3f}"))

- El **CV del bootstrap** coincide con el del método delta (diferencias de 4 a 7%) y es un poco menor. Son dos cálculos independientes: uno analítico de segundo orden, con la fórmula de **Prasad-Rao**, y otro por simulación. Que coincidan valida ambos. Que el analítico sea algo mayor es lo esperable, porque el término 2·g3 tiende a ser conservador.
- El sesgo es despreciable: menos de 1% relativo en todos los casos. La corrección analítica de la transformación inversa está haciendo su trabajo.
- La verificación de la unión da 0 exacto.

El hallazgo: σ²_u es frágil incluso en el agrupado. En 21% de las réplicas de d principal (30% sin degeneradas) la REML cayó en la frontera, y en R entre 10% y 17%. Con la σ²_u estimada (0.00029 en d) tan pequeña frente a la ψ media (0.0034), la distribución muestral del estimador tiene mucha masa en cero. Dicho de otro modo, con estos datos, una muestra de la ENVIPE ligeramente distinta habría producido un EBLUP puramente sintético para d una de cada cinco veces. Tres consecuencias:

1. Tener lista la alternativa ajustada no era una precaución teórica: el bootstrap la usó en 1,127 de 8,000 réplicas.
2. El ECM del bootstrap ya incorpora esa fragilidad, porque cada réplica aplica la misma regla. Los intervalos son honestos al respecto.
3. El **Rao-Yu** agrega una segunda componente de varianza (la correlación temporal), así que el problema de la frontera va a ser más agudo, no menos. Conviene llegar con la alternativa ajustada ya programada para las dos componentes.

In [ ]:
# tabla de confiabilidad y eblup_resultados

# --- 1. Recuperar los umbrales exactos del día 5 a partir de sus clasificaciones
# No los reescribo de memoria: los deduzco de los rangos de CV de cada categoría
# y verifico que las categorías no se traslapen.
def umbrales_dia5(col):
    g = (directos.dropna(subset=["cv"]).groupby(col)["cv"]
         .agg(["min", "max", "median", "size"]).sort_values("median"))
    cortes = []
    for i in range(len(g) - 1):
        hi, lo_sig = g["max"].iloc[i], g["min"].iloc[i + 1]
        corte = round(round((hi + lo_sig) / 2 / 0.05) * 0.05, 2)   # múltiplo de 0.05
        assert hi <= corte <= lo_sig, f"{col}: categorías traslapadas cerca de {corte}"
        cortes.append(corte)
    return list(g.index), cortes, g

CRITERIOS = {}
for col in ["conf_inegi", "conf_builgil"]:
    etiquetas, cortes, g = umbrales_dia5(col)
    CRITERIOS[col] = (etiquetas, cortes)
    print(f"{col}: cortes de CV = {cortes}")
    print(g.to_string(float_format=lambda x: f"{x:.3f}"), "\n")

def clasificar(cv, col):
    etiquetas, cortes = CRITERIOS[col]
    return pd.cut(cv, [-np.inf] + cortes + [np.inf], labels=etiquetas, right=True)

# --- 2. Columnas finales en escala de proporción ------------------------------
fin = eb[eb["elegido"]].copy()
agr = fin["modelo"] == "agrupado"

# ECM en proporción: bootstrap en el agrupado, delta en los modelos por año
fin["ecm_p"] = np.where(agr, fin["ecm_p_boot"], fin["ecm_p_delta"])
fin["metodo_ecm_p"] = np.where(agr, "bootstrap", "delta")
fin.loc[fin["usa_directo"], ["ecm_p", "metodo_ecm_p"]] = [np.nan, "directo"]
fin.loc[fin["usa_directo"], "ecm_p"] = fin.loc[fin["usa_directo"], "ee"] ** 2
fin["cv_modelo"] = np.sqrt(fin["ecm_p"]) / fin["p_modelo"]

# Intervalos por año: simétricos en θ, retransformados (quedan asimétricos en p)
z = 1.959964
pa_ = ~agr
th, se = fin.loc[pa_, "theta_final"], np.sqrt(fin.loc[pa_, "ecm_theta_final"])
fin.loc[pa_, "ic95_inf"] = np.sin((th - z * se).clip(0, np.pi / 2)) ** 2
fin.loc[pa_, "ic95_sup"] = np.sin((th + z * se).clip(0, np.pi / 2)) ** 2

for col in ["conf_inegi", "conf_builgil"]:
    fin[col + "_modelo"] = clasificar(fin["cv_modelo"], col)

# --- 3. Tabla antes y después (especificaciones principales) ------------------
filas = []
antes = directos[directos["geografia"] == "ocurrencia"]
for col in ["conf_inegi", "conf_builgil"]:
    for p in ["d", "k", "R"]:
        conteo = antes.loc[antes["param"] == p, col].value_counts()
        filas += [{"criterio": col, "param": p, "momento": "1_directo",
                   "categoria": c, "celdas": int(n)} for c, n in conteo.items()]
        for mod in ["agrupado", "por_anio"]:
            s = fin[(fin["param"] == p) & (fin["especificacion"] == "principal")
                    & (fin["modelo"] == mod)]
            conteo = s[col + "_modelo"].value_counts()
            filas += [{"criterio": col, "param": p, "momento": f"2_eblup_{mod}",
                       "categoria": c, "celdas": int(n)} for c, n in conteo.items()]
tabla_conf = (pd.DataFrame(filas)
              .pivot_table(index=["criterio", "param", "momento"], columns="categoria",
                           values="celdas", aggfunc="sum", fill_value=0))
for col in ["conf_inegi", "conf_builgil"]:
    print(f"\n{col}")
    print(tabla_conf.loc[col, CRITERIOS[col][0]].to_string())

print("\nCeldas con CV > 30% (principal):")
for p in ["d", "k", "R"]:
    a = (antes.loc[antes["param"] == p, "cv"] > 0.30).sum()
    g_ = fin[(fin["param"] == p) & (fin["especificacion"] == "principal")]
    print(f"  {p}: directo {a:2d}  |  agrupado "
          f"{(g_.loc[g_['modelo'] == 'agrupado', 'cv_modelo'] > 0.30).sum():2d}  |  por año "
          f"{(g_.loc[g_['modelo'] == 'por_anio', 'cv_modelo'] > 0.30).sum():2d}")

# --- 4. Guardar entregables ----------------------------------------------------
COLS = ["param", "modelo", "especificacion", "ajuste_anio", "metodo", "alc", "nombre",
        "anio_ref", "tipo_celda", "n", "est", "ee", "cv", "theta_dir", "psi",
        "sintetico", "gamma", "theta_eblup", "ecm_theta", "usa_directo",
        "theta_final", "ecm_theta_final", "p_modelo", "ecm_p", "metodo_ecm_p",
        "cv_modelo", "ic95_inf", "ic95_sup", "sesgo_boot",
        "conf_inegi", "conf_inegi_modelo", "conf_builgil", "conf_builgil_modelo"]
eblup_resultados = fin[COLS].rename(columns={"est": "p_directo", "ee": "ee_directo",
                                             "cv": "cv_directo"})
eblup_resultados.to_parquet(RUTA_SALIDA / "eblup_resultados.parquet", index=False)
tabla_conf.to_csv(RUTA_SALIDA / "tabla_confiabilidad_antes_despues.csv",
                  encoding="utf-8-sig")
print(f"\neblup_resultados: {eblup_resultados.shape} → guardado")

- **R**: de 51 celdas con CV mayor a 30% se pasa a cero en el agrupado. Con el criterio de Buil-Gil, de 13 celdas confiables a 80 de 80. Con el del INEGI, 15 celdas pasan a precisión alta cuando antes no había ninguna.
- **d**: de 32 celdas con CV alto a cero; 77 de 80 con precisión alta según el INEGI.
- **k**: las 80 celdas con precisión alta, incluida la que no tenía estimación directa.
- Por año: la mejora es real pero parcial (7 celdas de R siguen por encima de 30%). La comparación con el agrupado es el argumento empírico para combinar años.

Los umbrales se recuperaron limpios: 15% y 30% para el INEGI, 25% y 50% para Buil-Gil. La categoría “degenerada” corresponde a las celdas con CV de 0 (k = 1); ninguna celda modelada cae ahí.

La mejora es condicional al modelo. El ECM, con fórmula o con bootstrap, mide la precisión suponiendo que el **Fay-Herriot** está bien especificado. El **bootstrap** genera los datos desde ese mismo modelo, así que no puede detectar si está mal. El supuesto más vulnerable del agrupado es justo el que corrige el **Rao-Yu**: trata las cinco observaciones de cada alcaldía como independientes. Si una alcaldía tiene un efecto que persiste en el tiempo, el agrupado no lo aprovecha, y su ECM puede ser optimista. Hoy la tabla demuestra que el método funciona bajo sus supuestos; Rao-Yu permite revisar el más débil. 

Bajo el modelo agrupado, ninguna celda supera el umbral.

In [ ]:
# Directo contra EBLUP (arriba) y CV directo contra CV modelado (abajo)

import matplotlib.pyplot as plt

g = eblup_resultados.query("modelo == 'agrupado' and especificacion == 'principal'")
TITULOS = {"d": "d = P(denuncia | delito)",
           "k": "k = P(carpeta | denuncia)",
           "R": "R = P(carpeta | delito)"}
gmax = g["gamma"].max()

fig, ax = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
for j, p in enumerate(["d", "k", "R"]):
    s = g[(g["param"] == p) & g["p_directo"].notna()]
    tam = 10 + 70 * s["n"] / s["n"].max()               # tamaño = muestra de la celda

    # --- Arriba: estimación. Sobre la diagonal, el modelo no movió nada;
    # lejos de ella, encogió hacia el sintético. Los puntos chicos (poca muestra,
    # γ bajo) deben ser los más alejados.
    a = ax[0, j]
    sc = a.scatter(s["p_directo"], s["p_modelo"], c=s["gamma"], s=tam,
                   cmap="viridis", vmin=0, vmax=gmax, edgecolor="k", linewidth=0.3)
    lim = [min(s["p_directo"].min(), s["p_modelo"].min()) - 0.02,
           max(s["p_directo"].max(), s["p_modelo"].max()) + 0.02]
    a.plot(lim, lim, ls="--", color="gray", lw=1)
    a.set_xlim(lim); a.set_ylim(lim)
    a.set_title(TITULOS[p])
    a.set_xlabel("Estimador directo")
    a.set_ylabel("EBLUP agrupado")

    # --- Abajo: precisión. Todo punto bajo la diagonal ganó precisión.
    s2 = s[s["cv_directo"].notna() & (s["cv_directo"] > 0)]
    b = ax[1, j]
    b.scatter(s2["cv_directo"], s2["cv_modelo"], c=s2["gamma"], s=tam[s2.index],
              cmap="viridis", vmin=0, vmax=gmax, edgecolor="k", linewidth=0.3)
    tope = s2["cv_directo"].max() * 1.05
    b.plot([0, tope], [0, tope], ls="--", color="gray", lw=1)
    for u, estilo in [(0.15, ":"), (0.30, "-.")]:
        b.axhline(u, color="firebrick", ls=estilo, lw=0.8)
        b.axvline(u, color="firebrick", ls=estilo, lw=0.8)
    b.set_xlim(0, tope); b.set_ylim(0, tope)
    b.set_xlabel("CV del directo")
    b.set_ylabel("CV del EBLUP (bootstrap)")
    b.text(0.04, 0.95,
           f"CV mediano: {s2['cv_directo'].median():.2f} → {s2['cv_modelo'].median():.2f}\n"
           f"CV > 30%: {(s2['cv_directo'] > 0.30).sum()} → {(s2['cv_modelo'] > 0.30).sum()}",
           transform=b.transAxes, va="top", fontsize=9,
           bbox=dict(facecolor="white", edgecolor="0.7", boxstyle="round"))

fig.colorbar(sc, ax=ax, shrink=0.6, label="γ: peso de la encuesta en el EBLUP")
fig.suptitle("Estimador directo contra EBLUP de Fay-Herriot agrupado "
             "(ocurrencia, 16 alcaldías × 5 años)\n"
             "Tamaño del punto: muestra de la celda. Líneas rojas: umbrales de CV de 15% y 30%",
             fontsize=11)

for ext in ["png", "pdf"]:
    fig.savefig(RUTA_SALIDA / f"fig_directo_vs_eblup.{ext}", dpi=200)
plt.show()

# Nota de decisiones: selección de covariables y EBLUP de Fay-Herriot

## Resumen

Bajo el Fay-Herriot agrupado, ninguna de las 240 celdas de ocurrencia supera 30% de CV; en los directos eran 92 (51 en `R`, 32 en `d`, 9 en `k`). El CV mediano de `R` baja de 0.332 a 0.164.

La selección dejó tres conjuntos distintos: `d` con SESNSP, rezago a 7 días y canal digital; `R` solo con canal digital; `k` sin covariables. El modelo agrupado con efectos fijos de año es el principal; los modelos por año quedan como comparación.

La mejora es condicional al modelo. El bootstrap genera los datos desde el mismo Fay-Herriot, así que no detecta errores de especificación; el supuesto más débil (independencia entre años de una misma alcaldía) se revisa con el Rao-Yu.

## Selección de covariables

Cada parámetro tiene su propio conjunto, elegido por todos los subconjuntos y confirmado con REML.

| Parámetro | Especificación principal | Sensibilidad | R² de la señal (REML) | Índice de condición (cov / con años) |
|---|---|---|---|---|
| `d` | `sesnsp_nucleo`, `prop_7d_std`, `prop_digital_robo` | sin degeneradas | 0.60 (0.67 sin degeneradas) | 1.9 / 5.1 |
| `k` | solo efectos de año | `log_densidad`, `dist_mp_km` | −0.12 con acceso | — / 4.2 |
| `R` | `prop_digital_robo` | `sesnsp_nucleo`, `prop_7d_std`, `prop_digital_robo` | 0.24 (0.43 con SESNSP) | 1.0 / 4.9 |

**Reconciliación con cifras previas.** La correlación de +0.72 entre densidad y SESNSP corresponde al año 2020; sobre las 80 filas es +0.61, y baja cada año (0.72 en 2020, 0.53 en 2024). El índice de condición de 10.9 de las ocho candidatas originales se reproduce con la convención de Belsley, que es la usada aquí.

**Pares prohibidos.** No se eliminaron variables de antemano: ningún subconjunto podía contener ambos miembros de un par con |r| > 0.70. Son cinco pares: `prop_fem_18`–`eje_socio_1`, `dist_mp_km`–`sesnsp_nucleo`, `prop_ocupada_12`–`agencias_mp_100k`, `eje_socio_1`–`dist_mp_km` y `log_densidad`–`sesnsp_nucleo`. El último se mantuvo por conservadurismo: supera 0.70 en 2020, y el modelo por año incluye ese año.

**Procedimiento.** Se recorrieron 230 subconjuntos de hasta 4 covariables, con efectos de año siempre incluidos y θ en escala arcoseno como variable dependiente. Las dos covariables de la Fiscalía se pasaron a escala z sobre las 80 filas.

**Criterios clásicos contra R² de la señal.** Cp, AIC y R² ajustado quedaron cerca de cero, porque el directo incluye el error muestral, que ninguna covariable explica. Se agregó el R² de la señal, 1 − σ²_u(conjunto) / σ²_u(solo años). Calculado por momentos estaba contaminado por las celdas degeneradas; con REML las clasificaciones se mantienen y las señales son más claras.

**Coeficientes de `d`.** El canal digital entra con signo positivo (t = +2.82). El SESNSP entra con signo negativo (t = −2.12) y actúa como supresor: sin él, el R² de la señal cae de 0.60 a 0.18.

**`k` sin covariables.** Las variables de acceso (densidad y distancia al MP) aumentan σ²_u en 11%. Ninguna covariable territorial detecta la variación de `k`; sus coeficientes (t ≈ −1.6) se reportan como evidencia, no como modelo.

**Variación temporal.** La fracción de la varianza que ocurre dentro de cada alcaldía es 8% para SESNSP, 24% para `prop_7d_std` y 45% para `prop_digital_robo`. En la práctica, el SESNSP se comporta como una covariable fija.

## Decisión de circularidad

La especificación principal de `R` va sin SESNSP; la versión con SESNSP se reporta en el cuerpo del trabajo como sensibilidad.

**No hay circularidad en la estimación.** `R` se estima con la ENVIPE, que es independiente del registro de la Fiscalía.

**Sí hay dependencia funcional en el mapa corregido.** El EBLUP es γ·directo + (1 − γ)·x'β, con γ mediano de 0.17 en `R`. Si x incluye SESNSP, que compila los mismos datos de la Fiscalía, en las celdas con poco peso de la encuesta queda R̂ ≈ a + b·C. El mapa C / R̂ se vuelve entonces una función de C cuya forma depende de b.

**El signo agrava el problema.** SESNSP entra a `R` con coeficiente negativo (t = −1.69), así que el mapa C / (a − b·C) exageraría las diferencias entre alcaldías por construcción.

**El costo no es cero.** Con REML, quitar SESNSP baja el R² de la señal de 0.43 a 0.24 y sube σ²_u de 0.00035 a 0.00047. La estimación por momentos sugería un costo de un punto de γ; esa cifra quedó descartada.

**Para `d` no aplica.** `d` no entra en el cociente C / R, así que conserva SESNSP en su especificación principal.

## Implementación del Fay-Herriot

El Fay-Herriot se programó directamente en Python y pasó las cuatro verificaciones.

**Varianza de trabajo.** ψ_d = `var_theta_asen` = 1 / (4·`n_ef_suav`), con efecto de diseño suavizado por parámetro y geografía (1.54, 1.59 y 1.48 en ocurrencia). Se verificó en las 480 filas. Suavizar es lo correcto, porque el modelo supone ψ_d conocida.

**Estimación de σ²_u.** Máxima verosimilitud restringida (REML) por scoring de Fisher, con valor inicial en la mediana de ψ, como `eblupFH`. La REML es a la ML lo que dividir entre n − p es a dividir entre n: el término log|X'V⁻¹X| descuenta el costo de estimar β.

$$
\ell_{REML}(\sigma^2_u) = -\tfrac{1}{2}\left[\log|V| + \log|X'V^{-1}X| + \theta' P \theta\right]
$$

**Frontera.** Antes de iterar se evalúa la derivada en σ²_u = 0; si es menor o igual a cero, el máximo está en la frontera. La alternativa es la REML ajustada de Li y Lahiri, que suma log σ²_u y garantiza una varianza estrictamente positiva. Regla: REML, salvo frontera; en ese caso, la ajustada.

**ECM.** g1 + g2 + 2·g3 para REML (Datta y Lahiri). Para la versión ajustada se usa la misma fórmula como aproximación; Li y Lahiri derivan un término adicional que falta verificar.

**Verificaciones.** El scoring coincide con una búsqueda en malla de 2,000 puntos (0.000289 contra 0.000291). En los 63 ajustes, el EBLUP quedó entre el directo y el sintético. En el agrupado, γ y ψ tienen correlación de Spearman −1 y el ECM es menor que ψ en todas las celdas.

**Discrepancia entre REML y momentos.** En `d`, la REML dio 0.00029 y el estimador de momentos 0.00176. La causa principal es la celda degenerada de Milpa Alta 2022 (`d` = 0 con n = 51, residual estandarizado −3.4): sin degeneradas, el estimador de momentos cae a 0.00042 y la REML apenas a 0.00018. El exceso de dispersión tiene mediana negativa en los tres tercios de ψ, así que la ψ suavizada no subestima la varianza de forma sistemática.

**Celdas degeneradas.** En la especificación principal entran al ajuste con su θ en 0 o π/2 y su ψ, según el tratamiento ya definido para esas celdas. Como sensibilidad, se excluyen y reciben la predicción sintética. Un θ en el borde viola la normalidad, que el arcoseno no alcanza a corregir.

## Resultados del EBLUP

El agrupado tiene entre 35% y 61% del ECM de los modelos por año, y el modelo pesa entre 80% y 90% en la mayoría de las celdas.

| Parámetro y especificación | σ²_u REML (agrupado) | γ mediano | ψ media | ECM medio agrupado | ECM medio por año | Agrupado / por año | Réplicas bootstrap en frontera |
|---|---|---|---|---|---|---|---|
| `d` principal | 0.00029 | 0.11 | 0.00335 | 0.00053 | 0.00137 | 0.39 | 214 de 1,000 |
| `k` principal | 0.00439 | 0.20 | 0.02321 | 0.00467 | 0.00766 | 0.61 | 63 de 1,000 |
| `k` sens_acceso | 0.00490 | 0.22 | 0.02321 | 0.00537 | 0.00955 | 0.56 | 48 de 1,000 |
| `R` principal | 0.00047 | 0.17 | 0.00324 | 0.00057 | 0.00110 | 0.52 | 99 de 1,000 |
| `R` sens_sesnsp | 0.00035 | 0.14 | 0.00324 | 0.00056 | 0.00160 | 0.35 | 175 de 1,000 |

Todo en escala θ; el ECM por año corresponde al método elegido.

**Frontera en los modelos por año.** Cayeron en la frontera 5 de los 25 ajustes por año: `R` principal 2023, `R` sens_sesnsp 2022 y 2023, y `k` (ambas especificaciones) 2023. La σ²_u por año de `d` va de 0.0001 a 0.0021, un factor de veinte entre años.

**Frontera en el bootstrap.** 1,127 de 8,000 réplicas cayeron en la frontera; en `d`, una de cada cinco. Con estos datos, una muestra ligeramente distinta habría dado con frecuencia un EBLUP puramente sintético para `d`. El ECM del bootstrap ya incorpora esa fragilidad.

**Regla de respaldo.** En cada celda se reporta el EBLUP si su ECM es menor que ψ, y el directo en caso contrario (columna `usa_directo`). Afecta 28 celdas, todas en los modelos por año y en el tercil de ψ baja, con una razón ECM/ψ mediana de 1.11 y máxima de 1.50. Con 16 áreas, el costo de estimar σ²_u supera la ganancia en las celdas con mucha muestra. El agrupado no tiene ninguna.

**Transformación inversa.** Se usa la media exacta de sin² bajo normalidad, con el ECM como varianza. La corrección va de 0.0004 a 0.0026 en la mediana del agrupado.

$$
\hat p = \tfrac{1}{2} - \tfrac{1}{2}\cos(2\hat\theta)\, e^{-2\,\mathrm{ECM}}
$$

**ECM e intervalos en proporción.** Para el agrupado: bootstrap paramétrico de González-Manteiga et al., con B = 1,000 y semilla 20261012, reajuste completo y la misma regla de frontera en cada réplica; intervalos básicos por percentiles. El CV del bootstrap coincide con el del método delta dentro de 4% a 7%, y el sesgo relativo es menor a 1%. Para los modelos por año: método delta, con intervalos simétricos en θ retransformados; ninguna celda final quedó cerca del borde.

## Confiabilidad antes y después

Con el agrupado, las 80 celdas de cada parámetro quedan como confiables por el criterio de Buil-Gil, y ninguna queda con reservas por el del INEGI.

| Parámetro | Momento | INEGI: alta (< 15%) | INEGI: tolerable (15–30%) | INEGI: con reservas (> 30%) | Buil-Gil: confiable (≤ 25%) | Buil-Gil: con cautela (25–50%) | Buil-Gil: no usable (> 50%) | Degenerada |
|---|---|---|---|---|---|---|---|---|
| `d` | Directo | 0 | 47 | 32 | 31 | 44 | 4 | 1 |
| `d` | EBLUP agrupado | 77 | 3 | 0 | 80 | 0 | 0 | 0 |
| `d` | EBLUP por año | 5 | 71 | 4 | 67 | 13 | 0 | 0 |
| `k` | Directo | 31 | 37 | 9 | 63 | 14 | 0 | 2 |
| `k` | EBLUP agrupado | 80 | 0 | 0 | 80 | 0 | 0 | 0 |
| `k` | EBLUP por año | 78 | 2 | 0 | 80 | 0 | 0 | 0 |
| `R` | Directo | 0 | 28 | 51 | 13 | 55 | 11 | 1 |
| `R` | EBLUP agrupado | 15 | 65 | 0 | 80 | 0 | 0 | 0 |
| `R` | EBLUP por año | 0 | 73 | 7 | 61 | 19 | 0 | 0 |

Ocurrencia, especificación principal. Umbrales recuperados de las clasificaciones de las estimaciones directas. En los directos de `k` hay 79 celdas: la que no tiene estimación recibe la predicción sintética en los modelos.

**El CV favorece artificialmente a `k`.** `k` es una proporción alta, y el CV divide el error entre la estimación. En escala arcoseno, su ψ media (0.023) es siete veces la de `d` o `R`: es el parámetro peor medido, aunque solo 9 celdas superaban 30% de CV. Su ECM en θ bajó de 0.023 a 0.0047, la mayor reducción absoluta de los tres. La misma distorsión explica que, en la figura, algunas celdas de `k` con directo cercano a 1 tengan CV modelado mayor que el directo: el modelo bajó su proporción, no perdió precisión.

## Hallazgos y advertencias

La mayor parte de la dispersión entre alcaldías que muestran los directos es ruido de muestreo: entre 60% y 73% de la varianza residual, según el parámetro.

**Los mapas modelados son mucho más planos que los directos.** En la figura, `k` va de 0.33 a 1.0 en los directos y queda aproximadamente entre 0.60 y 0.75 en el EBLUP. `R` pasa de 0–0.24 a cerca de 0.05–0.10, y `d` de 0–0.30 a cerca de 0.08–0.17. Es un resultado, no un defecto, y hay que comunicarlo como tal.

**Implicación para el argumento de `k`.** Se esperaba que la variación de `k` entre alcaldías revelara incumplimiento con distribución territorial. Bajo el modelo, la variación territorial de `k` es modesta y ninguna covariable de acceso la explica. Lo que sí es grande es el nivel: si `k` ronda 0.68, cerca de un tercio de las denuncias no termina en carpeta, pese a la obligación del CNPP. Pregunta abierta: ¿el argumento se reformula como una barrera institucional extendida en toda la ciudad, más que concentrada en ciertas alcaldías?

**La mejora en confiabilidad es condicional al modelo.** El ECM, analítico o por bootstrap, supone que el Fay-Herriot está bien especificado. El agrupado trata las cinco observaciones de cada alcaldía como independientes; si hay efectos persistentes, su ECM puede ser optimista. Redacción sugerida: "bajo el modelo agrupado, ninguna celda supera el umbral".

**La σ²_u de `d` y `R` es pequeña y frágil.** En `d` es 0.00029 frente a una ψ media de 0.0034, y el bootstrap cae en la frontera en 21% de las réplicas. Las conclusiones sobre la variación propia de cada alcaldía en `d` deben formularse con cautela.

## Pendientes, archivos y referencias

**Para el Rao-Yu**

- Programar la versión ajustada para las dos componentes de varianza: con una componente más, la frontera será más frecuente.
- Partir de las mismas especificaciones; en `k`, toda la variación temporal recaerá en los efectos aleatorios y de año.
- Comparar el ECM del Rao-Yu con el del agrupado para medir cuánto optimismo tenía el supuesto de independencia.
- Verificar el término adicional del ECM de la REML ajustada.
- Decidir el encuadre del hallazgo de `k` (nivel contra variación territorial).

**Archivos generados** (en `RUTA_SALIDA`)

| Archivo | Contenido |
|---|---|
| `seleccion_covariables_todos.csv` | Los 230 subconjuntos × 3 parámetros, con Cp, AIC, R² ajustado y R² de la señal por momentos |
| `seleccion_covariables_finalistas.csv` | Finalistas con σ²_u, AIC e índices de condición |
| `seleccion_final.csv` | Las cinco especificaciones congeladas |
| `seleccion_revision_reml.csv` | Revisión de la selección con REML, con y sin degeneradas |
| `eblup_theta_crudo.parquet` | Los 63 ajustes en escala θ, con ambos métodos |
| `eblup_resumen_ajustes.csv` | Resumen por ajuste: σ²_u, frontera, γ, ECM y verificaciones |
| `eblup_resultados.parquet` | 1,040 filas × 33 columnas: método elegido, escala de proporción, ECM, CV, intervalos y confiabilidad |
| `tabla_confiabilidad_antes_despues.csv` | La tabla de confiabilidad |
| `fig_directo_vs_eblup.png` / `.pdf` | Directo contra EBLUP y CV directo contra CV modelado |